# Apply LSTM model to hydro data

Use HydroDF csv that was downloaded and compiled in [LSTM_data](LSTM_data.ipynb). Use torch310 environment.

In [3]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib

from supportingscripts import LSTM_helper

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In this application, the LSTM uses a 30-day lookback window of meteorological and hydrologic inputs to predict daily streamflow. By learning relationships across time, the model can represent processes such as snowmelt timing, precipitation-runoff response, and seasonal dynamics, making it well-suited for hydrologic forecasting tasks.


Data splitting:
- Training: Diamond Fork near Thistle (10149400), Weber River near Oakley (10128500), and East Canyon Creek at I-80 (10133650)
- Validation: East Canyon Creek near Jeremy Ranch (10133800)
- Testing: East Canyon Creek near Morgan (10133980)

In [ ]:
# Append HydroDFs into a single df
Hydro_df = pd.DataFrame()
for i in range(1, 6):
    df = pd.read_csv(f"../files/HydroDF/HydroDF_{i}.csv")
    Hydro_df = pd.concat([Hydro_df, df], ignore_index=False)
    Hydro_df.sort_values(by='Date', inplace=True)
# Convert 'Date' to datetime and set as index
Hydro_df['Date'] = pd.to_datetime(Hydro_df['Date'])
Hydro_df.set_index('Date', inplace=True)

#save df
Hydro_df.to_csv("../data/HydroDF/HydroDF.csv")

Hydro_df.head()

In [ ]:
# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Data parameters
FILE_PATH = "data/HydroDF/HydroDF.csv"
SITE_COL = "site_no"
DATE_COL = "Date"
TARGET_COL = "flow_cms"

# Hyperparameters
LOOKBACK_DAYS = 30 #number of past time steps to use for prediction
BATCH_SIZE = 64 #number of samples per batch for training
EPOCHS = 50 #number of times to iterate over the entire training dataset
PATIENCE = 8 #number of epochs to wait for improvement in validation loss before stopping training
LEARNING_RATE = 1e-3 #step size for updating model parameters during training

# Define the SITES for splitting the dataset into training, validation, and testing sets
TRAIN_SITES = (10149400, 10128500, 10133650)
VAL_SITE = 10133800
TEST_SITE = 10133980


### Load and inspect data

In [ ]:
# Load and preprocess the data
df = pd.read_csv(FILE_PATH)

# Clean column names, removing spaces and special characters
clean_cols = []
for c in df.columns:
    c = str(c).strip().replace('"', '')
    c = ''.join(ch if ch.isalnum() else '_' for ch in c)
    while '__' in c:
        c = c.replace('__', '_')
    c = c.strip('_')
    clean_cols.append(c)
df.columns = clean_cols

# Convert date column to datetime and sort by date
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

print('Rows:', len(df))
print('Date range:', df[DATE_COL].min().date(), 'to', df[DATE_COL].max().date())
print('Years:', sorted(df[DATE_COL].dt.year.unique()))
df.head()

### Choose predictors

In [ ]:
# Identify numeric feature columns, excluding target and date columns
exclude_cols = {TARGET_COL, SITE_COL, 'Date', 'station_id', 'Unnamed_0'}

# We want to include only numeric columns as features, and exclude any non-numeric or irrelevant columns
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

print('Target column:', TARGET_COL)
print('Number of features:', len(feature_cols))
print(feature_cols)

### Choose feature columns

If necessary, may not need to reduce the number of features the model looks at.

In [ ]:
# choose features
feature_cols =['TUM_SWE_cm', 'DAN_SWE_cm', 'SLI_SWE_cm']
cols = feature_cols + [TARGET_COL]

#select these columns from the dataframe
df = df[[SITE_COL] + cols]
df.head()

### Fill missing values

In [ ]:
# Check for missing values, fill them using linear interpolation, forward fill, and backward fill
df[feature_cols + [TARGET_COL]] = (
    df[feature_cols + [TARGET_COL]]
    .interpolate(method='linear', limit_direction='both')
    .ffill()
    .bfill()
)

### Data split

Split data by site into training, validation, and testing data sets

In [ ]:
# Split the data into training, validation, and testing sets based on site numbers

train_df = df[df['site_no'].isin(TRAIN_SITES)].copy()
val_df = df[df['site_no'].isin(VAL_SITE)].copy()
test_df = df[df['site_no'].isin(TEST_SITE)].copy()

print('Train rows:', len(train_df))
print('Validation rows:', len(val_df))
print('Test rows:', len(test_df))

### Fit scalars on training data

In [ ]:
# Scale features and target using MinMaxScaler, fit on training data only to prevent data leakage
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit the scalers on the training data only to prevent data leakage
feature_scaler.fit(train_df[feature_cols])
target_scaler.fit(train_df[[TARGET_COL]])

#save scalers for later use
joblib.dump(feature_scaler, "model/LSTM/feature_scaler.pkl")
joblib.dump(target_scaler, "model/LSTM/target_scaler.pkl")

#add scaled columns to dataframes
train_scaled = LSTM_helper.add_scaled_columns("model/LSTM/", feature_cols, TARGET_COL, train_df)
val_scaled = LSTM_helper.add_scaled_columns("model/LSTM/", feature_cols, TARGET_COL, val_df)
test_scaled = LSTM_helper.add_scaled_columns("model/LSTM/", feature_cols, TARGET_COL, test_df)

In [ ]:
train_scaled.head()

### Build 30-day sequences

In [ ]:
#create sequences for LSTM
X_train, y_train, d_train = LSTM_helper.make_sequences(DATE_COL, train_scaled, LOOKBACK_DAYS, feature_cols, TARGET_COL)
X_val, y_val, d_val = LSTM_helper.make_sequences(DATE_COL, val_scaled, LOOKBACK_DAYS, feature_cols, TARGET_COL)
X_test, y_test, d_test = LSTM_helper.make_sequences(DATE_COL, test_scaled, LOOKBACK_DAYS, feature_cols, TARGET_COL)

print('Train shape:', X_train.shape, y_train.shape)
print('Val shape:', X_val.shape, y_val.shape)
print('Test shape:', X_test.shape, y_test.shape)

# make sure the shapes are correct for # of time steps and features (second and third numbers)

### Pytorch

In [ ]:
#create dataloaders, this converts the numpy arrays into PyTorch tensors and creates batches for training
train_loader = DataLoader(LSTM_helper.SequenceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(LSTM_helper.SequenceDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(LSTM_helper.SequenceDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

### Define model

In [ ]:
# Define the LSTM model, loss function, and optimizer
model = LSTM_helper.LSTMRegressor(input_size=len(feature_cols), hidden_size=64, num_layers=1, dropout=0.0)
# Move the model to the appropriate device (GPU if available, otherwise CPU)
model = model.to(device)
# Define the loss function (Mean Squared Error) and the optimizer (Adam)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

### Training loop

In [ ]:
# Train the model with early stopping based on validation loss
best_val_loss = np.inf #initialize best validation loss to infinity so that any improvement will be detected
best_state = None #variable to store the model state with the best validation loss
patience_counter = 0 #counter to track how many epochs have passed without improvement in validation loss
history = {'train_loss': [], 'val_loss': []} #dictionary to store training and validation loss history for each epoch

# Loop over the number of epochs specified, training the model and evaluating on the validation set at each epoch
for epoch in range(1, EPOCHS + 1):
    model.train() # Set the model to training mode, which enables dropout and batch normalization layers to behave appropriately during training
    batch_losses = [] # List to store the loss for each batch, which will be averaged to get the epoch loss
    for xb, yb in train_loader: # Loop over the training data in batches, where xb is the input batch and yb is the corresponding target batch
        xb = xb.to(device) # Move the input batch to the appropriate device (GPU if available, otherwise CPU)
        yb = yb.to(device) # Move the target batch to the appropriate device (GPU if available, otherwise CPU)
        optimizer.zero_grad() # Clear the gradients of all optimized parameters before the backward pass
        pred = model(xb) # Forward pass: compute the model's predictions for the input batch
        loss = criterion(pred, yb) # Compute the loss between the model's predictions and the true target values using the defined loss function (Mean Squared Error)
        loss.backward() # Backward pass: compute the gradients of the loss with respect to the model parameters
        optimizer.step() # Update the model parameters using the computed gradients and the defined optimization algorithm (Adam)
        batch_losses.append(loss.item()) # Append the loss for the current batch to the list of batch losses

# After processing all batches in the training set, compute the average training loss for the epoch and evaluate the model on the validation set
    train_loss = float(np.mean(batch_losses)) # Compute the average training loss for the epoch by taking the mean of the batch losses
    val_loss, _, _ = LSTM_helper.evaluate(model, criterion, device, val_loader) # Evaluate the model on the validation set using the defined evaluation function, which returns the validation loss and other metrics
    history['train_loss'].append(train_loss) # Append the average training loss for the epoch to the history dictionary for later analysis and visualization
    history['val_loss'].append(val_loss) # Append the validation loss for the epoch to the history dictionary for later analysis and visualization

    print(f'Epoch {epoch:03d} | train loss = {train_loss:.5f} | val loss = {val_loss:.5f}')

# Check if the validation loss has improved compared to the best validation loss seen so far. If it has improved, update the best validation loss and save the current model state. If it has not improved, increment the patience counter. If the patience counter exceeds the defined patience threshold, trigger early stopping and exit the training loop.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else: # If the validation loss has not improved, increment the patience counter
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print('Early stopping triggered.')
            break
# After training is complete, check if a best model state was saved during training (i.e., if the validation loss improved at least once). If a best model state exists, load it into the model to ensure that we have the best performing model based on validation loss for subsequent evaluation on the test set.
if best_state is not None:
    model.load_state_dict(best_state)